# Preparação consolidada para `4_analise_etapas_1_7.ipynb`

Este notebook substitui os blocos necessários de `2_analise_graficos.ipynb` e `3_analise_assuntos.ipynb`. Ele gera somente os dois arquivos que o notebook 4 realmente consome:

- `output/dataset_analise_questoes.csv`;
- `output/mapeamento_provas_questoes.json`.

Execute este notebook depois de gerar os arquivos de misconceptions e as métricas da Etapa 4, e antes de executar `4_analise_etapas_1_7.ipynb`. As provas são lidas diretamente de `Etapa_2/output/provas_catalogadas`; nenhuma cópia em `Etapa_5` é necessária.


In [ ]:
from pathlib import Path
import json
import re
import pandas as pd

BASE = Path.cwd()
OUTPUT_DIR = BASE / "output"
OUTPUT_DIR.mkdir(exist_ok=True)
MISC_PATH = OUTPUT_DIR / "misconceptions_resumo_por_questao.csv"
METRICS_PATH = BASE.parent / "Etapa_4" / "output" / "metricas_questoes_com_dificuldade.csv"
# Fonte canônica: não copiar nem reorganizar as provas na Etapa 5.
# O notebook 4 e as etapas anteriores já usam este mesmo diretório.
PROVAS_PATH = BASE.parent / "Etapa_2" / "output" / "provas_catalogadas"
QUESTIONS_OUT = OUTPUT_DIR / "dataset_analise_questoes.csv"
STRUCTURE_DIR = OUTPUT_DIR
STRUCTURE_OUT = STRUCTURE_DIR / "mapeamento_provas_questoes.json"

print("Diretório de trabalho:", BASE)
print("Misconceptions:", MISC_PATH)
print("Métricas de questões:", METRICS_PATH)
print("Provas:", PROVAS_PATH)


In [ ]:
def merge_questions_and_metrics(misc_path, metrics_path, output_path):
    """Combina o resumo de misconceptions com as métricas agregadas das questões."""
    if not misc_path.exists():
        raise FileNotFoundError(f"Arquivo ausente: {misc_path}")
    if not metrics_path.exists():
        raise FileNotFoundError(f"Arquivo ausente: {metrics_path}")

    misc = pd.read_csv(misc_path)
    metrics = pd.read_csv(metrics_path)

    misc_key = 'question' if 'question' in misc.columns else 'questao_id' if 'questao_id' in misc.columns else 'id'
    metrics_key = 'question' if 'question' in metrics.columns else 'questao_id' if 'questao_id' in metrics.columns else 'id'

    misc[misc_key] = misc[misc_key].astype(str).str.strip()
    metrics[metrics_key] = metrics[metrics_key].astype(str).str.strip()

    merged = misc.merge(
        metrics,
        left_on=misc_key,
        right_on=metrics_key,
        how='inner',
        suffixes=('_misc', '_metr')
    )

    if misc_key != 'question' and 'question' not in merged.columns:
        merged = merged.rename(columns={misc_key: 'question'})
    elif 'question_misc' in merged.columns:
        merged = merged.rename(columns={'question_misc': 'question'})

    # Quando as duas fontes possuem a mesma coluna, preserva a versão do resumo
    # de misconceptions e remove a cópia criada pelo merge.
    for column in list(merged.columns):
        if column.endswith('_metr'):
            base_name = column[:-5]
            misc_name = base_name + '_misc'
            if misc_name in merged.columns:
                merged = merged.drop(columns=[column])
                if base_name not in merged.columns:
                    merged = merged.rename(columns={misc_name: base_name})

    if 'question' not in merged.columns:
        raise ValueError("Não foi possível obter a coluna question após o merge.")

    merged['question'] = pd.to_numeric(merged['question'], errors='coerce')
    merged = merged.dropna(subset=['question']).copy()
    merged['question'] = merged['question'].astype(int)
    merged.to_csv(output_path, index=False)
    print(f"dataset_analise_questoes.csv gerado: {len(merged)} linhas x {len(merged.columns)} colunas")
    return merged

questions_df = merge_questions_and_metrics(MISC_PATH, METRICS_PATH, QUESTIONS_OUT)


In [ ]:
def parse_data_file(path):
    """Extrai os mesmos metadados e exercícios usados pelo notebook 3 original."""
    text = path.read_text(encoding='utf-8', errors='replace')
    patterns = {
        'assessment_title': r'assessment title:\s*(.+)',
        'class_name': r'class name:\s*(.+)',
        'class_number': r'class number:\s*(\d+)',
        'start': r'start:\s*(.+)',
        'end': r'end:\s*(.+)',
        'language': r'language:\s*(.+)',
        'type': r'type:\s*(.+)',
        'weight': r'weight:\s*(\d+)',
        'total_exercises': r'total_exercises:\s*(\d+)',
    }
    info = {}
    for key, pattern in patterns.items():
        match = re.search(pattern, text, flags=re.IGNORECASE)
        if match:
            info[key] = match.group(1).strip()

    exercises = {}
    for exercise_number, content in re.findall(r'exercise (\d+):\s*(.+)', text, flags=re.IGNORECASE):
        exercises[f'exercise_{exercise_number}'] = [q.strip() for q in content.split(' or ') if q.strip()]
    info['exercises'] = exercises
    return info

def identify_module_exam(path):
    """Reproduz a identificação robusta de módulo e prova do notebook 3."""
    parts = path.parts
    module = None
    exam = None
    keywords_module = ('introducao', 'programacao', 'algoritmos', 'estrutura')
    keywords_exam = ('avaliacao', 'prova', 'tp', 'parcial', 'final')

    for i, part in enumerate(parts[:-1]):
        lower = part.lower()
        if any(keyword in lower for keyword in keywords_module):
            module = part
            for candidate in parts[i + 1:-1]:
                candidate_lower = candidate.lower()
                if candidate_lower in {'exam', 'test', 'avaliacao'}:
                    continue
                if any(keyword in candidate_lower for keyword in keywords_exam):
                    exam = candidate
                    break
            if exam is None and len(parts) >= 2:
                exam = parts[-2]
            break

    if module is None and len(parts) >= 3:
        module = parts[-3]
        exam = parts[-2]
    return module, exam

def build_structure(provas_path):
    """Cria o mesmo contrato JSON consumido pelo notebook 4."""
    if not provas_path.exists():
        raise FileNotFoundError(f"Diretório de provas ausente: {provas_path}")

    mapping = {}
    processed = 0
    ignored = 0
    for path in provas_path.rglob('*.data'):
        if any(part.lower() == 'excecao' for part in path.parts):
            ignored += 1
            continue
        module, exam = identify_module_exam(path)
        if not module or not exam:
            continue
        info = parse_data_file(path)
        file_id = path.stem
        mapping.setdefault(module, {}).setdefault(exam, {})[file_id] = {
            'caminho': str(path),
            'info': info,
        }
        processed += 1

    mapped_questions = set()
    for exams in mapping.values():
        for files in exams.values():
            for record in files.values():
                for questions in record.get('info', {}).get('exercises', {}).values():
                    mapped_questions.update(str(q).strip() for q in questions)
    csv_questions = set(str(q).strip() for q in questions_df['question'])

    structure = {
        'mapeamento_provas': mapping,
        'estatisticas': {
            'total_modulos': len(mapping),
            'total_provas': sum(len(exams) for exams in mapping.values()),
            'total_arquivos': processed,
            'questoes_csv': len(questions_df),
            'questoes_mapeadas': len(mapped_questions.intersection(csv_questions)),
            'arquivos_ignorados_excecao': ignored,
        },
    }
    STRUCTURE_DIR.mkdir(parents=True, exist_ok=True)
    STRUCTURE_OUT.write_text(json.dumps(structure, ensure_ascii=False, indent=2, default=str), encoding='utf-8')
    print(f"mapeamento_provas_questoes.json gerado: {processed} arquivos .data; {ignored} ignorados em excecao")
    return structure

structure = build_structure(PROVAS_PATH)


In [ ]:
# Validação mínima das saídas consumidas pelo notebook 4.
required_questions = {'question', 'respostas'}
missing_questions = required_questions - set(questions_df.columns)
if missing_questions:
    raise ValueError(f"dataset_analise_questoes.csv sem colunas obrigatórias: {sorted(missing_questions)}")

if not STRUCTURE_OUT.exists():
    raise FileNotFoundError(STRUCTURE_OUT)

structure_loaded = json.loads(STRUCTURE_OUT.read_text(encoding='utf-8'))
if 'mapeamento_provas' not in structure_loaded:
    raise ValueError("mapeamento_provas_questoes.json não contém mapeamento_provas")

print("Validação concluída.")
print("Saídas prontas para 4_analise_etapas_1_7.ipynb:")
print(" -", QUESTIONS_OUT)
print(" -", STRUCTURE_OUT)


## Ordem de execução

1. Execute o parser `1_Misconceptions_Parser.py` ou o notebook equivalente.
2. Execute este notebook consolidado.
3. Verifique as mensagens de validação e os dois arquivos de saída.
4. Execute `4_analise_etapas_1_7.ipynb`.

Os notebooks antigos `2_analise_graficos.ipynb` e `3_analise_assuntos.ipynb` não precisam mais ser executados para preparar o notebook 4. Não use `Etapa_5/Provas` nem `Etapa_5/Provas_Arrumadas` como fontes de dados; elas são cópias históricas e devem permanecer vazias ou ser removidas.
